# Testing ADAPT.jl interface in python

In [1]:
import subprocess
import os
from datetime import datetime
from pathlib import Path
import json
from itertools import islice 
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# cur_input_graphs_dict = json.load(open('graphs_n10_1idx_jl_022625_dict.json'))

In [4]:
# cur_input_graphs_dict['Graph_4_n10_021225']

In [5]:
from adapt_utils import (
    run_adapt_jl_parallel,
    show_adapt_logs,
    get_combined_res_df
)

In [6]:
adapt_gpt_dir = Path(
    "/home/mrzaizai2k/code_bao/ADAPT_GPT"
)
adapt_output_dir = "./ADAPT.jl_results/qaoa_python_test"
n_graphs = 5
n_nodes = 10
input_graph_filename = "ADAPT.jl_results/graphs_n10.json"


In [7]:
import networkx as nx
import random

def add_weights_to_nx_graph(G, weighted=True, use_negative=False):
    elist = []
    for u, v in G.edges():
        if weighted:
            w = random.uniform(0.1, 1.0)
            if use_negative and random.random() < 0.5:
                w *= -1
        else:
            w = -1 if (use_negative and random.random() < 0.5) else 1

        elist.append([int(u)+1, int(v)+1, float(round(w, 2))])  
        # +1 to match Julia 1-indexing
    return elist

In [8]:
def generate_graphs(
    n_graphs=10,
    n_nodes=10,
    density=None,          # if None → random
    weighted=True,
    use_negative=False
):
    graphs_dict = {}

    for i in range(n_graphs):
        if density is None:
            p = random.uniform(0.6, 0.9)   # random density
        else:
            p = density

        G = nx.erdos_renyi_graph(n=n_nodes, p=p)

        # avoid empty graph
        while G.number_of_edges() == 0:
            G = nx.erdos_renyi_graph(n=n_nodes, p=p)

        elist = add_weights_to_nx_graph(G, weighted, use_negative)

        graph_name = f"Graph_{i}_n{n_nodes}"

        graphs_dict[graph_name] = {
            "elist": elist,
            "n_nodes": n_nodes
        }

    return graphs_dict

In [9]:
import json

def save_graphs_to_json(graphs_dict, filename):
    with open(filename, "w") as f:
        json.dump(graphs_dict, f, indent=2)

In [10]:
def load_graphs(filename):
    with open(filename, "r") as f:
        return json.load(f)

In [11]:


graphs = generate_graphs(
    n_graphs=n_graphs,
    n_nodes=n_nodes,
    density=None,          # or e.g. 0.7
    weighted=True,
    use_negative=False
)

save_graphs_to_json(graphs, input_graph_filename)

# Load back
cur_input_graphs_dict = load_graphs(input_graph_filename)

# Access like your image
print(cur_input_graphs_dict["Graph_0_n10"])

{'elist': [[1, 2, 0.34], [1, 3, 0.75], [1, 4, 0.81], [1, 5, 0.43], [1, 7, 0.52], [1, 8, 0.19], [1, 9, 0.63], [1, 10, 0.99], [2, 3, 0.16], [2, 4, 0.95], [2, 5, 0.36], [2, 6, 0.11], [2, 7, 0.53], [2, 8, 0.61], [2, 9, 0.71], [2, 10, 0.64], [3, 4, 0.43], [3, 6, 0.6], [3, 7, 0.42], [3, 8, 0.93], [3, 9, 0.71], [3, 10, 0.44], [4, 5, 0.46], [4, 6, 0.23], [4, 7, 0.13], [4, 9, 0.42], [4, 10, 0.91], [5, 7, 0.3], [5, 8, 0.7], [5, 9, 0.61], [5, 10, 0.87], [6, 8, 0.12], [6, 9, 0.64], [6, 10, 0.76], [7, 9, 1.0], [7, 10, 0.11], [8, 9, 0.92], [8, 10, 0.89]], 'n_nodes': 10}


In [12]:
logs_list, cur_proc = run_adapt_jl_parallel(
    script_dir=adapt_gpt_dir,
    output_dir=adapt_output_dir,
    input_graphs=adapt_gpt_dir / input_graph_filename,
    n_workers=2,
    graphs_number=n_graphs,
    trials_per_graph=1,
    max_params=50,
    gamma_0="gamma0_grid.json",
    pool_name="qaoa_double_pool",
    use_floor_stopper=True,
    temp_folder="temp_data",
)

Splitting input graphs into 2 parts
Starting worker 0 on node: DESKTOP-H2CRQMR
Starting worker 1 on node: DESKTOP-H2CRQMR


In [13]:
# show_adapt_logs(logs_list, n_lines=20)

In [14]:
# show_adapt_logs(logs_list, n_lines=20, pbar_only=True)

In [47]:
combined_res_df = get_combined_res_df(
    adapt_output_dir,
    debug_limit=5,
)

Reading ADAPT.jl results from:
	 /home/mrzaizai2k/code_bao/ADAPT_GPT/qaoa_python_test


Opening ADAPT results (qaoa_python_test): 100%|██████████| 1/1 [00:00<00:00, 259.93it/s]


df_list len: 1


Opening graphs (qaoa_python_test): 100%|██████████| 1/1 [00:00<00:00, 748.58it/s]

df_list len: 1
Graphs count:
g_method
input_file    2
Name: count, dtype: int64
Aggregating results...


In [48]:
combined_res_df.columns

Index(['graph_num', 'run', 'prefix', 'method', 'optimizer', 'gamma0',
       'pooltype', 'graph_name', 'n_nodes', 'energy_list', 'took_time',
       'energy_mqlib', 'op_list', 'approx_ratio', 'edge_weight_norm_coef',
       'β_coeff', 'γ_coeff', 'coeff', 'energy_eigen', 'cut_mqlib', 'cut_adapt',
       'cut_eig', 'state_vect_adapt', 'success_flag', 'g_method',
       'edgelist_json', 'H_frob_norm', 'worker_id', 'edgelist_list',
       'edgelist_list_len', 'num_connected_comp', 'n_layers', 'graph_id'],
      dtype='object')

In [49]:
temp_df = combined_res_df.drop(columns=["prefix", "energy_list", "coeff", "cut_mqlib", "cut_adapt", "cut_eig", "state_vect_adapt", "graph_id"])

In [51]:
temp_df[:1]['approx_ratio']

0    0.979537
Name: approx_ratio, dtype: float64

In [50]:
temp_df[:1]

,graph_num,run,method,optimizer,gamma0,pooltype,graph_name,n_nodes,took_time,energy_mqlib,op_list,approx_ratio,edge_weight_norm_coef,β_coeff,γ_coeff,energy_eigen,success_flag,g_method,edgelist_json,H_frob_norm,worker_id,edgelist_list,edgelist_list_len,num_connected_comp,n_layers
0,1,1,qaoa,BFGS,0.01,qaoa_double_pool,Graph_2_n10,10,4.756007,-12.95,"[127, 162, 191, 178, 171, 70, 19, 27]",0.979537,1.0,"[0.7853972201372746, -0.7853978958439035, 0.7853979639581304, -0.7853987795735375, -0.7854136956002163, 0.7854007134314032, 0.7853980273725946, 0.7853951290016465]","[-2.757052208520373e-06, 9.257543603958474e-07, 3.2710546838178687e-06, 4.672626809903444e-05, -8.88780203381037e-05, 9.597408651857956e-05, -0.0002396216234901, 0.0001597080314604]",-12.95,True,input_file,"[[1,2,0.65],[1,3,0.86],[2,3,0.51],[2,4,0.77],[3,4,0.25],[1,5,0.26],[2,5,0.9],[2,6,0.44],[3,6,0.19],[4,6,0.96],[1,7,0.29],[3,7,0.21],[6,7,0.54],[2,8,0.3],[3,8,0.34],[4,8,0.71],[5,8,0.42],[6,8,0.88],[1,9,0.55],[3,9,0.94],[4,9,0.99],[5,9,0.25],[7,9,0.41],[8,9,0.76],[1,10,0.38],[2,10,0.5],[3,10,0.66],[5,10,0.91],[6,10,0.91],[9,10,0.82]]",286.517696,worker_unknown_pid_8164_ts_26-03-27__14_03_graphs_json,"[[1, 2, 0.65], [1, 3, 0.86], [2, 3, 0.51], [2, 4, 0.77], [3, 4, 0.25], [1, 5, 0.26], [2, 5, 0.9], [2, 6, 0.44], [3, 6, 0.19], [4, 6, 0.96], [1, 7, 0.29], [3, 7, 0.21], [6, 7, 0.54], [2, 8, 0.3], [3, 8, 0.34], [4, 8, 0.71], [5, 8, 0.42], [6, 8, 0.88], [1, 9, 0.55], [3, 9, 0.94], [4, 9, 0.99], [5, 9, 0.25], [7, 9, 0.41], [8, 9, 0.76], [1, 10, 0.38], [2, 10, 0.5], [3, 10, 0.66], [5, 10, 0.91], [6, 10, 0.91], [9, 10, 0.82]]",30,1,8


We can read graph from edgelist_json, use our model to create data and comapre with approx_ratio, compare n_layers